# CrossFuse v5 - Stage D2: Attribution ablation (what actually fixed cross-dataset?)

> **Status (2026-09): A4, A13, A14, A15 ran; `X1` was not run (optional).** Combined results: `results/ablation_attribution_v5.csv`. Mean cross-dataset AUC: A4 0.851, A13 0.846, A15 0.750 (time-limited), A14 0.550 -> the FF++ *pretraining* stage is the ingredient; mixing FF++ into fine-tuning adds nothing (A4 ~ A13). See `README.md` ("What made the difference").

`C-eval` showed cross-dataset video AUC going 0.61 -> 0.855. Three things changed at once, so
the honest question is **which one did it**:

| Arm | Backbone | FF++ in the training mix | FF++ encoder pretraining | Question it answers |
|---|---|---|---|---|
| `A4_full_crossfuse` | CLIP ViT-L/14 | yes | yes | Reference: the full recipe under THIS notebook's reduced epoch budget |
| `A13_no_ffpp_data` | CLIP ViT-L/14 | **no** | yes | Does mixing FF++ into training matter, given the pretrained encoder? |
| `A14_efficientnet_backbone` | **EfficientNet-B4** | yes | no (ViT weights cannot load) | Does the FF++ data alone fix it without CLIP? |
| `A15_no_ffpp_pretrain` | CLIP ViT-L/14 | yes | **no** | Does the dedicated FF++ pretraining stage matter, or just mixing FF++ in? |
| `X1_clip_no_ffpp_at_all` *(optional)* | CLIP ViT-L/14 | **no** | **no** | Is it just CLIP itself? (plain OpenAI CLIP on FakeAVCeleb only) |

**The metric that matters is zero-shot cross-dataset video AUC on Celeb-DF v2 and DFDC** (the same
pools `C-eval` used), NOT in-domain FakeAVCeleb AUC -- every arm will score ~0.99 in-domain, so
that column cannot tell them apart. In-domain AUC is still logged as a sanity check.

**Single seed, reduced budget (2 frozen + up to 8 fine-tune epochs, patience 4).** Bootstrap CIs on
these pools are roughly +/-0.05, so treat differences smaller than ~0.05 as "no evidence of a
difference", not as a ranking. Arms are only comparable to each other, not to the fully trained
`B-train` model.

## How to run (each arm is a full training run -- do NOT try all in one session)
Edit `ARMS_TO_RUN` in cell 2, then Save Version -> Save & Run All (Commit). Suggested sessions:
1. `["A14_efficientnet_backbone", "A13_no_ffpp_data"]` -- cheap (EfficientNet, and A13 trains on ~4x fewer rows)
2. `["A15_no_ffpp_pretrain"]`
3. `["A4_full_crossfuse"]`
4. *(optional)* `["X1_clip_no_ffpp_at_all"]`

Each session writes its own `ablation_attrib_<arms>.csv`; download them all and send them over to
be combined. A per-arm wall-clock budget (see `plan_arm_budget_s`) stops training cleanly with the
best checkpoint if an arm runs long -- the stop reason and epochs run are recorded, because an arm
cut short is NOT a fair comparison and must be flagged, not silently reported.

**Attach:** `crossfuse-v5-lib`, `crops-v5`, `ffpp-crops-v5`, `ffpp-encoder-v5`, a Celeb-DF v2
dataset (`Celeb-real`, `YouTube-real`, `Celeb-synthesis`) and the DFDC sample (with `metadata.json`).
Internet ON, Environment latest, GPU T4.

In [ ]:
import time
_T0 = time.time()   # session clock -- per-arm time budgets are sized from this

# open_clip's real deps, installed --no-deps so pip cannot disturb Kaggle's torch/torchvision
# pairing (see B-train / B1 cell 1 for the full reasoning).
!pip install -q --no-deps open_clip_torch timm safetensors ftfy regex pyyaml
!pip install -q --no-deps facenet-pytorch

import sys, glob, os
_lib_hits = glob.glob("/kaggle/input/**/crossfuse_v5.py", recursive=True)
assert _lib_hits, ("crossfuse_v5.py not found under /kaggle/input -- attach the "
                   "crossfuse-v5-lib dataset in Add Input, then restart the session.")
sys.path.insert(0, os.path.dirname(_lib_hits[0]))

import torch, torchvision
print(f"torch {torch.__version__} | torchvision {torchvision.__version__}")
from torchvision.models import efficientnet_b0  # noqa: F401
import open_clip  # noqa: F401
print("torchvision.models.efficientnet_b0 -- OK | open_clip -- OK")

import csv, copy, json
import numpy as np
from crossfuse_v5 import (
    CONFIG, run_training_pipeline, evaluate_deterministic, safe_auc, bootstrap_ci,
    merge_manifests, build_detector, extract_face_crops_contiguous, make_rng,
    mc_logits, device,
)
print("Using crossfuse_v5 library from:", _lib_hits[0])
print("device:", device, "| GPUs:", torch.cuda.device_count())

In [ ]:
# ================= EDIT THIS PER SESSION =================
ARMS_TO_RUN = ["A14_efficientnet_backbone", "A13_no_ffpp_data"]
SESSION_HOURS = 8.0        # Kaggle's cap is 9h; the rest is safety margin
# =========================================================

ABLATION_EPOCHS = dict(EPOCHS_FROZEN=2, EPOCHS_FINETUNE=8, PATIENCE=4)
N_CROSS_FRAMES = 32        # same contiguous-run length C-eval extracts per video

# Single-GPU only: nn.DataParallel is ~3x SLOWER for CLIP ViT-L/14 (see CLAUDE.md).
CONFIG["MULTI_GPU"] = False


def plan_arm_budget_s(session_h, elapsed_s, n_arms_left, eval_s_per_arm=1500.0,
                      safety_s=1800.0, floor_s=600.0):
    # Seconds of TRAINING one arm may use so all remaining arms still fit the session.
    # Reserves per-arm time for cross-dataset scoring and a final safety margin.
    remaining = session_h * 3600.0 - elapsed_s - safety_s - eval_s_per_arm * n_arms_left
    return max(floor_s, remaining / max(1, n_arms_left))


# ---- FakeAVCeleb manifest (picked by header, not glob order) ----
CROP_DIR = None
for p in glob.glob("/kaggle/input/**/manifest.csv", recursive=True):
    with open(p) as f:
        if "audio_label" in f.readline():
            CROP_DIR = os.path.dirname(p)
            break
assert CROP_DIR, "FakeAVCeleb manifest.csv not found -- attach crops-v5."
with open(os.path.join(CROP_DIR, "manifest.csv")) as f:
    FAV_ROWS = list(csv.DictReader(f))
for r in FAV_ROWS:
    r["video_label"] = int(r["video_label"]); r["audio_label"] = int(r["audio_label"])
    r["fps"] = float(r["fps"])

# ---- FF++ manifest ----
FFPP_DIR, FFPP_ROWS = None, []
for p in glob.glob("/kaggle/input/**/manifest.csv", recursive=True):
    with open(p) as f:
        head = f.readline()
    if "method" in head and "target_id" in head:
        FFPP_DIR = os.path.dirname(p)
        break
if FFPP_DIR:
    with open(os.path.join(FFPP_DIR, "manifest.csv")) as f:
        FFPP_ROWS = list(csv.DictReader(f))
    for r in FFPP_ROWS:
        r["label"] = int(r["label"])

ENC = next(iter(glob.glob("/kaggle/input/**/ffpp_encoder_v5.pth", recursive=True)), None)
print(f"FakeAVCeleb {len(FAV_ROWS)} rows | FF++ {len(FFPP_ROWS)} rows | "
      f"encoder {'yes' if ENC else 'NO'}")

# Two manifests so an arm can drop FF++ data while everything else is held fixed.
MANIFEST_MERGED = (merge_manifests(FAV_ROWS, CROP_DIR, FFPP_ROWS, FFPP_DIR,
                                   seed=CONFIG["SEED"], verbose=False)
                   if FFPP_ROWS else None)
MANIFEST_FAV = [dict(r, crop_dir=CROP_DIR, has_audio=1) for r in FAV_ROWS]

In [ ]:
# ---- Cross-dataset pools, extracted ONCE and shared by every arm ----
# Same pools as C-eval (DFDC: every labelled video in metadata.json; Celeb-DF v2: shuffled with
# make_rng("celebdf_eval"), first 400), so numbers here are directly comparable to C-eval's.
# Face detection (MTCNN) is the slow part; caching the crops means each arm only pays for a
# forward pass. ~5 GB of RAM for ~800 videos -- fine on Kaggle's 30 GB.
import json as _json

dfdc_entries = []
for meta_path in glob.glob("/kaggle/input/**/metadata.json", recursive=True):
    folder = os.path.dirname(meta_path)
    with open(meta_path) as f:
        data = _json.load(f)
    for vid_name, info in data.items():
        full_path = os.path.join(folder, vid_name)
        if os.path.exists(full_path) and info.get("label", "").upper() in ("REAL", "FAKE"):
            dfdc_entries.append((full_path, 0 if info["label"].upper() == "REAL" else 1))

celebdf_entries = []
for sub, lbl in (("Celeb-real", 0), ("YouTube-real", 0), ("Celeb-synthesis", 1)):
    for vpath in glob.glob(f"/kaggle/input/**/{sub}/**/*.mp4", recursive=True):
        celebdf_entries.append((vpath, lbl))
_rng = make_rng("celebdf_eval"); _rng.shuffle(celebdf_entries)
celebdf_entries = celebdf_entries[:400]

# Fail loudly: an ablation with no cross-dataset data cannot answer its own question.
assert dfdc_entries, "No DFDC videos found -- attach the DFDC sample WITH its metadata.json."
assert celebdf_entries, "No Celeb-DF v2 videos found -- attach a Celeb-DF v2 dataset."
print(f"DFDC {len(dfdc_entries)} videos | Celeb-DF {len(celebdf_entries)} videos")

CROSS_CACHE = {}
_detector = build_detector(str(device))
for ds_name, entries in (("dfdc", dfdc_entries), ("celebdf", celebdf_entries)):
    cached = []
    for i, (vpath, label) in enumerate(entries):
        crops, valid, *_ = extract_face_crops_contiguous(
            vpath, _detector, rng=make_rng(vpath), n_frames=N_CROSS_FRAMES,
            out_size=CONFIG["CROP_SIZE"])
        if crops is None or int(valid.sum()) == 0:
            continue
        cached.append((crops, valid, label))
        if (i + 1) % 100 == 0:
            print(f"  {ds_name}: {i + 1}/{len(entries)} extracted")
    CROSS_CACHE[ds_name] = cached
    print(f"{ds_name}: {len(cached)}/{len(entries)} videos usable")
del _detector
torch.cuda.empty_cache()
print(f"cross-dataset crop cache ready after {(time.time() - _T0) / 60:.1f} min")

In [ ]:
# ---- Arm definitions ----
BASE = dict(BACKBONE="clip_vit_l14", TRAIN_RES=224,
            PRETRAINED_ENCODER=ENC, BATCH_CLIPS=4, GRAD_ACCUM=3)


def cfg_variant(**overrides):
    c = copy.deepcopy(CONFIG)
    c.update(ABLATION_EPOCHS)
    c.update(overrides)
    return c


ARMS = {
    "A4_full_crossfuse":
        (cfg_variant(**BASE), MANIFEST_MERGED),
    "A13_no_ffpp_data":            # FF++ removed from the mix; FF++-pretrained encoder kept
        (cfg_variant(**BASE), MANIFEST_FAV),
    "A14_efficientnet_backbone":   # ViT checkpoint cannot load into EfficientNet -> no encoder
        (cfg_variant(BACKBONE="efficientnet_b4", TRAIN_RES=224, PRETRAINED_ENCODER=None,
                     FREEZE_BLOCKS=4), MANIFEST_MERGED),
    "A15_no_ffpp_pretrain":        # FF++ mixed in, but no dedicated FF++ pretraining stage
        (cfg_variant(**dict(BASE, PRETRAINED_ENCODER=None)), MANIFEST_MERGED),
    "X1_clip_no_ffpp_at_all":      # plain OpenAI CLIP on FakeAVCeleb only: "is it just CLIP?"
        (cfg_variant(**dict(BASE, PRETRAINED_ENCODER=None)), MANIFEST_FAV),
}

# Hard prerequisites, checked up front so a session cannot burn hours on a meaningless arm.
for name in ARMS_TO_RUN:
    assert name in ARMS, f"unknown arm {name!r}; choose from {list(ARMS)}"
    cfg, manifest = ARMS[name]
    if manifest is None:
        raise AssertionError(f"{name} needs the FF++ manifest -- attach ffpp-crops-v5.")
    if name in ("A4_full_crossfuse", "A13_no_ffpp_data") and ENC is None:
        raise AssertionError(f"{name} needs the FF++-pretrained encoder -- attach ffpp-encoder-v5.")
print("arms this session:", ARMS_TO_RUN)

In [ ]:
# ---- Cross-dataset scoring: video head, every window of the cached run, averaged ----
def score_cached(model, cached):
    W = CONFIG["WINDOW_FRAMES"]
    dummy_mfcc = torch.zeros(1, 40, CONFIG["AUDIO_FRAMES"]).to(device)
    y_true, y_score = [], []
    for crops, valid, label in cached:
        probs = []
        for s in range(0, max(1, crops.shape[0] - W + 1), W):
            win, vm = crops[s:s + W], valid[s:s + W]
            if win.shape[0] < W:
                pad = W - win.shape[0]
                win = np.concatenate([win, np.zeros((pad, *win.shape[1:]), np.uint8)])
                vm = np.concatenate([vm, np.zeros(pad, bool)])
            if not vm.any():
                continue
            x = torch.from_numpy(np.ascontiguousarray(win)).unsqueeze(0).to(device)
            m = torch.from_numpy(vm).unsqueeze(0).to(device)
            got = mc_logits(model, x, dummy_mfcc, None, m, n_samples=CONFIG["N_MC_SAMPLES"])
            probs.append(float(1.0 / (1.0 + np.exp(-got["video"].mean()))))
        if probs:
            y_true.append(label)
            y_score.append(float(np.mean(probs)))
    auc = safe_auc(y_true, y_score)
    lo, hi = bootstrap_ci(y_true, y_score, safe_auc)
    return auc, lo, hi, len(y_true)


FIELDS = ["arm", "backbone", "n_train_rows", "n_ffpp_rows", "encoder", "best_epoch",
          "train_minutes", "budget_min", "stopped_by_time",
          "indomain_test_video_auc", "indomain_test_fusion_auc",
          "dfdc_auc", "dfdc_lo", "dfdc_hi", "dfdc_n",
          "celebdf_auc", "celebdf_lo", "celebdf_hi", "celebdf_n", "mean_cross_auc"]
_tag = "_".join(n.split("_")[0] for n in ARMS_TO_RUN)
CSV_PATH = f"/kaggle/working/ablation_attrib_{_tag}.csv"
rows = []


def _write_csv():
    with open(CSV_PATH, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS, extrasaction="ignore")
        w.writeheader()
        w.writerows(rows)

In [ ]:
# ---- Run the arms ----
for k, name in enumerate(ARMS_TO_RUN):
    cfg, manifest = ARMS[name]
    budget_s = plan_arm_budget_s(SESSION_HOURS, time.time() - _T0, len(ARMS_TO_RUN) - k)
    cfg = dict(cfg, TIME_BUDGET_S=budget_s)
    print(f"\n{'=' * 64}\n  {name}  | train budget {budget_s / 60:.0f} min\n{'=' * 64}")

    t0 = time.time()
    result = run_training_pipeline(manifest, CROP_DIR, cfg, tag=name,
                                   seed=CONFIG["SEED"], verbose=True)
    train_s = time.time() - t0
    model = result["model"]

    ind = evaluate_deterministic(model, result["test_loader"], cfg)
    dfdc = score_cached(model, CROSS_CACHE["dfdc"])
    cdf = score_cached(model, CROSS_CACHE["celebdf"])

    row = dict(
        arm=name, backbone=cfg["BACKBONE"], n_train_rows=len(result["train_idx"]),
        n_ffpp_rows=sum(1 for r in manifest if r.get("corpus") == "ffpp"),
        encoder=os.path.basename(cfg["PRETRAINED_ENCODER"] or "none"),
        best_epoch=result["best_epoch"],
        train_minutes=round(train_s / 60, 1), budget_min=round(budget_s / 60, 1),
        stopped_by_time=(result.get("stop_reason") == "time_budget"),
        indomain_test_video_auc=ind["video_auc"], indomain_test_fusion_auc=ind["fusion_auc"],
        dfdc_auc=dfdc[0], dfdc_lo=dfdc[1], dfdc_hi=dfdc[2], dfdc_n=dfdc[3],
        celebdf_auc=cdf[0], celebdf_lo=cdf[1], celebdf_hi=cdf[2], celebdf_n=cdf[3],
        mean_cross_auc=(dfdc[0] + cdf[0]) / 2)
    rows.append(row)
    _write_csv()
    print(f"\n{name}: DFDC {dfdc[0]:.4f} [{dfdc[1]:.3f},{dfdc[2]:.3f}] | "
          f"Celeb-DF {cdf[0]:.4f} [{cdf[1]:.3f},{cdf[2]:.3f}] | "
          f"mean {row['mean_cross_auc']:.4f} | in-domain video {ind['video_auc']:.4f}")
    del result, model
    torch.cuda.empty_cache()

print(f"\nDone. Wrote {CSV_PATH}")

In [ ]:
# ---- Summary for this session ----
print(f"{'arm':28s} {'DFDC':>8s} {'CelebDF':>8s} {'mean':>7s} {'in-dom':>7s} {'best_ep':>7s} {'min':>6s}  flag")
for r in rows:
    flag = "CUT SHORT BY TIME BUDGET - not a fair comparison" if r["stopped_by_time"] else ""
    print(f"{r['arm']:28s} {r['dfdc_auc']:8.4f} {r['celebdf_auc']:8.4f} "
          f"{r['mean_cross_auc']:7.4f} {r['indomain_test_video_auc']:7.4f} "
          f"{r['best_epoch']:7d} {r['train_minutes']:6.1f}  {flag}")

print(f'''
Reference (NOT comparable -- fully trained 23-epoch B-train model, C-eval MC-dropout scoring):
  DFDC 0.8649 | Celeb-DF 0.8447 | mean 0.8548        Paper baseline (EfficientNet-B4): mean 0.6099

How to read the arms once all sessions are done (differences under ~0.05 are within noise):
  A14 ~ A4        -> the FF++ DATA fixed it; CLIP added little.
  A14 << A4       -> the CLIP backbone matters.
  A13 << A4       -> mixing FF++ into training matters (given the pretrained encoder).
  A15 << A4       -> the dedicated FF++ pretraining stage matters, not just mixing FF++ in.
  X1 ~ A4         -> it is just CLIP; the FF++ work added little.
  X1 << A4        -> CLIP alone does not do it; FF++ is doing real work.
''')